# Day 55: Python type hints - basics to advanced
**Python 3.12+**. Run top to bottom. Standard library only; no setup or network needed.

Type hints describe the values your code expects and returns. They help editors, readers and static checkers catch mistakes before execution. **Python does not enforce hints automatically.**

Start with sections 1-6; continue to reusable and advanced patterns when comfortable.

## 1. Variables and functions
`name: str` describes an input; `-> str` describes the return value. `-> None` means no useful value is returned. Hints do not convert values.

In [ ]:
customer: str = "Asha"
quantity: int = 2
unit_price: float = 19.5
active: bool = True

def line_total(price: float, count: int) -> float:
    return price * count

def welcome(name: str) -> str:
    return f"Hello, {name}!"

def show_message(message: str) -> None:
    print(message)

show_message(welcome(customer))
print(line_total(unit_price, quantity))
assert line_total(19.5, 2) == 39.0

### Hints versus runtime validation
The next call deliberately violates a hint, but Python still executes it. A static checker should flag the call. Validation requires real executable checks.

In [ ]:
def repeat_twice(value: int) -> int:
    return value * 2

print(repeat_twice(4))
print(repeat_twice("Hi"))  # Intentional type error for a checker; prints HiHi.

def require_positive(value: int) -> int:
    if type(value) is not int or value <= 0:
        raise ValueError("Expected a positive integer")
    return value

try:
    require_positive(-2)
except ValueError as error:
    print("Validation:", error)

## 2. Collections
Annotate the contents as well as the container. `tuple[str, int]` has two fixed positions; `tuple[int, ...]` has any number of integers.

In [ ]:
names: list[str] = ["Asha", "Ravi"]
stock: dict[str, int] = {"hub": 12, "mouse": 0}
tags: set[str] = {"new", "sale"}
row: tuple[str, int] = ("hub", 12)
counts: tuple[int, ...] = (2, 4, 6)

def available(items: dict[str, int]) -> list[str]:
    return [name for name, count in items.items() if count > 0]

print(available(stock))
print(row, counts)

## 3. Unions, None and defaults
`int | str` allows either type. `str | None` allows a string or `None` (older spelling: `Optional[str]`). Allowing `None` does **not** make an argument optional to supply; a default does that.

An `isinstance` or `None` check narrows the type for the code that follows.

In [ ]:
def normalize_id(value: int | str) -> str:
    if isinstance(value, int):
        return f"O{value}"
    return value.strip().upper()

def greeting(name: str | None = None) -> str:
    if name is None:
        return "Hello, guest"
    return f"Hello, {name.upper()}"

print(normalize_id(1001), normalize_id(" o1002 "))
print(greeting(), greeting("Asha"))

## 4. Any versus object
`Any` largely disables checking for that value. `object` can hold any Python value but requires narrowing before type-specific operations. Prefer `object` for unknown input you intend to inspect.

In [ ]:
from typing import Any

unchecked: Any = "hello"
unknown: object = "hello"
print(unchecked.upper())
if isinstance(unknown, str):
    print(unknown.upper())

# unchecked.missing_method() would pass many static checks but fail at runtime.
# unknown.upper() needs a narrowing check first.

## 5. Data records: TypedDict and dataclass
`TypedDict` describes dictionary keys to a checker; values remain ordinary dictionaries. A dataclass creates objects with attributes and a generated initializer. Neither automatically validates input types.

`NotRequired` permits an absent key; it is different from a present key whose value can be `None`.

In [ ]:
from typing import TypedDict, NotRequired
from dataclasses import dataclass

class OrderRow(TypedDict):
    order_id: str
    quantity: int
    note: NotRequired[str]

order_row: OrderRow = {"order_id": "O1001", "quantity": 2}

@dataclass
class Product:
    name: str
    price: float

product = Product("USB hub", 19.5)
print(order_row.get("note", "No note"))
print(product.name, product.price)
assert isinstance(order_row, dict)

## 6. Literal choices, aliases and distinct IDs
`Literal` limits accepted values for a checker. An alias gives an existing type a shorter name. `NewType` distinguishes otherwise similar values such as customer IDs and order IDs during static checking; it adds no runtime validation.

The `type` alias statement below requires Python 3.12+.

In [ ]:
from typing import Literal, NewType

type Status = Literal["pending", "shipped"]
type SalesRow = tuple[str, float]
CustomerId = NewType("CustomerId", int)
OrderId = NewType("OrderId", int)

def status_message(status: Status) -> str:
    return f"Status: {status}"

def load_customer(customer_id: CustomerId) -> str:
    return f"Customer {customer_id}"

sale: SalesRow = ("hub", 39.0)
print(status_message("shipped"), sale)
print(load_customer(CustomerId(101)))
# load_customer(OrderId(101))  # Static checker: wrong kind of ID.

## 7. Accept capabilities instead of concrete containers
Use `Iterable` when you only loop, `Sequence` when you also need indexing, and `Mapping` for read access by key. These accept more inputs than a specific `list` or `dict`.

`Iterator[int]` describes a stream yielding integers.

In [ ]:
from collections.abc import Iterable, Iterator, Mapping, Sequence

def sum_counts(values: Iterable[int]) -> int:
    return sum(values)

def lookup_stock(items: Mapping[str, int], name: str) -> int:
    return items.get(name, 0)

def count_up(limit: int) -> Iterator[int]:
    for number in range(limit):
        yield number

print(sum_counts([1, 2, 3]), sum_counts((4, 5)))
print(sum_counts(count_up(4)))
print(lookup_stock(stock, "hub"))

## 8. Generics preserve relationships
A generic type parameter connects input and output types. `first(list[str])` returns `str`; `first(list[int])` returns `int`. Using `Any` would lose that connection.

This is Python 3.12 syntax. Older code uses `TypeVar` and `Generic` for these patterns.

In [ ]:
def first[T](items: Sequence[T]) -> T:
    if not items:
        raise ValueError("Expected at least one item")
    return items[0]

@dataclass
class Box[T]:
    value: T

    def get(self) -> T:
        return self.value

print(first([10, 20]), first(["Asha", "Ravi"]))
box: Box[str] = Box("report.csv")
print(box.get())

### Why list types are usually invariant
A `list[Dog]` cannot safely be used as `list[Animal]`: a function accepting `list[Animal]` might append a Cat. For read access, `Sequence[Animal]` can accept a sequence of Dogs. This restriction prevents unsafe mutation.

In [ ]:
class Animal:
    pass

class Dog(Animal):
    pass

def count_animals(animals: Sequence[Animal]) -> int:
    return len(animals)

dogs: list[Dog] = [Dog(), Dog()]
print(count_animals(dogs))

## 9. Callable and async return types
`Callable[[int], str]` is a function taking an integer and returning a string. On `async def`, annotate the result obtained **after awaiting**; an async function used as a callback returns an `Awaitable` to its caller.

In [ ]:
import asyncio
from collections.abc import Awaitable, Callable

def apply_label(value: int, formatter: Callable[[int], str]) -> str:
    return formatter(value)

def format_order(value: int) -> str:
    return f"O{value}"

async def fetch_label(value: int) -> str:
    await asyncio.sleep(0)
    return format_order(value)

async_loader: Callable[[int], Awaitable[str]] = fetch_label
print(apply_label(1001, format_order))
print(await async_loader(1002))  # Jupyter supports top-level await.

## 10. Protocol: describe required behavior
A Protocol defines the methods an object must provide. Matching classes need not inherit from it. This is useful when swapping real services for small test implementations.

In [ ]:
from typing import Protocol

class Sender(Protocol):
    def send(self, message: str) -> str: ...

class ConsoleSender:
    def send(self, message: str) -> str:
        return f"Sent: {message}"

def notify(sender: Sender, message: str) -> str:
    return sender.send(message)

print(notify(ConsoleSender(), "Order ready"))

## 11. Overloads describe input-dependent results
Overload signatures help checkers select the return type. They do not implement runtime dispatch: the final function must handle every supported case.

In [ ]:
from typing import overload

@overload
def clean(value: str) -> str: ...

@overload
def clean(value: list[str]) -> list[str]: ...

def clean(value: str | list[str]) -> str | list[str]:
    if isinstance(value, str):
        return value.strip()
    return [item.strip() for item in value]

print(clean(" hub "))
print(clean([" hub ", " mouse "]))

## 12. Custom narrowing with TypeGuard
A guard tells a checker what is true when it returns `True`. Its implementation must genuinely check the claim. Here an immutable tuple avoids mutation surprises after narrowing.

In [ ]:
from typing import TypeGuard

def is_text_tuple(value: object) -> TypeGuard[tuple[str, ...]]:
    return isinstance(value, tuple) and all(isinstance(item, str) for item in value)

payload: object = ("sales", "stock")
if is_text_tuple(payload):
    print(", ".join(payload))
assert not is_text_tuple(("sales", 3))

## 13. Decorators without losing the signature
`ParamSpec` carries a function's parameter types through a wrapper; `TypeVar` carries its return type. `wraps` preserves runtime metadata. A plain `Callable[..., Any]` would discard useful checking.

In [ ]:
from functools import wraps
from typing import ParamSpec, TypeVar

P = ParamSpec("P")
R = TypeVar("R")

def traced(function: Callable[P, R]) -> Callable[P, R]:
    @wraps(function)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        print("Calling", function.__name__)
        return function(*args, **kwargs)
    return wrapper

@traced
def multiply(value: int, *, factor: int = 2) -> int:
    return value * factor

print(multiply(5, factor=3))
assert multiply(2) == 4

## 14. Self, Final and ClassVar
`Self` tracks the actual class in chainable methods, including subclasses. `Final` asks checkers to reject reassignment. `ClassVar` marks a class-level attribute; dataclasses exclude it from instance fields. These hints do not lock values at runtime.

In [ ]:
from typing import Self, Final, ClassVar

MAX_RETRIES: Final[int] = 3

@dataclass
class Query:
    table: str = "orders"
    engine: ClassVar[str] = "demo"

    def from_table(self, table: str) -> Self:
        self.table = table
        return self

query = Query().from_table("products")
print(query.table, Query.engine, MAX_RETRIES)

## 15. Metadata, casts and runtime inspection
`Annotated` attaches metadata that a framework may interpret. Python itself does not enforce it. `cast()` asserts a type to the checker; it neither checks nor converts the value.

Use real validation at external boundaries. `get_type_hints()` resolves annotations; only inspect trusted code because resolution can evaluate expressions.

In [ ]:
from typing import Annotated, cast, get_type_hints

def reserve(quantity: Annotated[int, "must be positive"]) -> str:
    if quantity <= 0:
        raise ValueError("Quantity must be positive")
    return f"Reserved {quantity}"

print(reserve(2))
print(get_type_hints(reserve, include_extras=True))
raw_value: object = "123"
claimed_number = cast(int, raw_value)
print("After cast:", type(claimed_number).__name__)  # Still str!
converted_number = int("123")
print("After conversion:", type(converted_number).__name__)

## 16. Exhaustiveness with assert_never
After handling every literal choice, `assert_never` helps a checker catch a missing case when choices grow. If execution reaches it, it raises an error too.

In [ ]:
from typing import assert_never

def next_action(status: Status) -> str:
    match status:
        case "pending":
            return "Pack order"
        case "shipped":
            return "Track delivery"
        case _:
            assert_never(status)

print(next_action("pending"))

## 17. Mini program: validate a row, then use typed data
External data needs runtime checks. After validation, a typed record makes the rest of the program easier to reason about. Prices use integer cents to avoid floating-point money arithmetic.

In [ ]:
class SaleInput(TypedDict):
    product: str
    quantity: int
    unit_price_cents: int

def parse_sale(raw: Mapping[str, object]) -> SaleInput:
    name = raw.get("product")
    count = raw.get("quantity")
    price = raw.get("unit_price_cents")
    if not isinstance(name, str) or not name.strip():
        raise ValueError("product must be a nonempty string")
    if not isinstance(count, int) or isinstance(count, bool) or count <= 0:
        raise ValueError("quantity must be a positive integer")
    if not isinstance(price, int) or isinstance(price, bool) or price < 0:
        raise ValueError("unit_price_cents must be a nonnegative integer")
    return {"product": name, "quantity": count, "unit_price_cents": price}

def invoice_total(sale: SaleInput) -> int:
    return sale["quantity"] * sale["unit_price_cents"]

sale_record = parse_sale({"product": "hub", "quantity": 2, "unit_price_cents": 1950})
print(sale_record, "Total cents:", invoice_total(sale_record))
assert invoice_total(sale_record) == 3900
try:
    parse_sale({"product": "hub", "quantity": "two", "unit_price_cents": 1950})
except ValueError as error:
    print("Invalid input:", error)

## 18. Try a static checker: one pass and one failure
Running Python checks runtime behavior. **mypy examines the code without running it.** These optional commands create two tiny scripts and check their annotations.

Run the blocks in order in a **Bash terminal** (Git Bash, WSL or Linux/macOS), not a Python cell or PowerShell. In Git Bash, you can first run `cd /c/Course/DataEng/Day55`. Use `python3` instead of `python` throughout if that is your interpreter command.

### A. Install mypy
Use the same Python interpreter to install and run the checker:

```bash
python -m pip install mypy
python -m mypy --version
```

### B. Create a valid script with tee
`tee` writes the file and displays its contents. The quoted `PY` marker keeps Bash from expanding the Python text. Rerunning a block replaces that demo file.

```bash
tee typehint_valid.py <<'PY'
def double(value: int) -> int:
    return value * 2

result: int = double(5)
print(result)
PY
```

Check it:

```bash
python -m mypy --strict typehint_valid.py
echo "Checker exit code: $?"
```

Expected: `Success: no issues found in 1 source file`, followed by exit code **0**.

Run the program separately:

```bash
python typehint_valid.py
```

Expected runtime output: `10`.

### C. Create an invalid script with tee
The function promises an integer input, but this caller supplies a string:

```bash
tee typehint_invalid.py <<'PY'
def double(value: int) -> int:
    return value * 2

result: int = double("5")
print(result)
PY
```

Check it:

```bash
python -m mypy --strict typehint_invalid.py
echo "Checker exit code: $?"
```

Expected diagnostic (format may vary by mypy version):

```text
typehint_invalid.py:4: error: Argument 1 to "double" has incompatible type "str"; expected "int" [arg-type]
Found 1 error in 1 file (checked 1 source file)
Checker exit code: 1
```

Read `$?` immediately after the checker: it is the previous command's exit status. If your shell uses `set -e`, the intentional failure may stop the shell before `echo` runs.

### D. Compare with runtime behavior

```bash
python typehint_invalid.py
```

Python prints `55`: string multiplication is valid at runtime, and hints do not enforce the intended integer contract. **The static checker catches the mistake before you run the code.** Change `double("5")` to `double(5)`, then rerun mypy to make the check pass.

**Practice:** add a `cancelled` Status and update `next_action`; make a `Box[int]`; add a `discount_cents` field and validate it in the mini program.

**References:** [Python typing](https://docs.python.org/3/library/typing.html), [typing guide and specification](https://typing.python.org/), [mypy getting started](https://mypy.readthedocs.io/en/stable/getting_started.html).
